In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_PATH = Path.cwd().parent
RELATIV_PATH_PROCESSED_DATA = "data/processed/processed_dataset_2023.csv"
RELATIV_PATH_INTERIM_DATA = "data/interim/interim_dataset.csv"

# Block 4 – Standardization and extreme values before clustering

## Decisions of blocks 1, 2 and 3: features selected for clustering (main version, 162 countries, 2023)

| Feature | Role | Log | Standardization |
|---|---|---|---|
| GDP per capita (PPP) | Active | yes | yes, after log |
| Life expectancy | Active | no | yes |
| CO₂ per capita | Active | yes | yes, after log |
| PM2.5 | Active | yes | yes, after log |
| Urbanization (% of population) | Active | no | yes |
| Population | Illustrative | no | no |
| Land area | Illustrative | no | no |
| Continent | Illustrative (later) | no | no |
| Health spending (% of GDP) | Removed from main version (variant, session 06) | no | n/a |
| Health spending per capita | Removed (built from GDP per capita) | n/a | n/a |

_Order: log first, then standardization. Standardization is applied at modeling time, not stored in `processed/`._

In [ ]:
MIN_POPULATION = 300000
COLUMNS_WITH_LOG = [
    "gdp_per_capita_ppp_current_ppp",
    "co2_emissions_per_capita_t",
    "pm2_5_exposure_mcg_m3",
]
Z_THRESHOLD = 2.5


def prepare_data(min_population=MIN_POPULATION, excluded_areas=()):
    """Load the processed dataset, exclude micro-states and listed countries, add log columns (raw columns are kept for the variants)."""
    df = pd.read_csv(PROJECT_PATH / RELATIV_PATH_PROCESSED_DATA)
    df = df[(df["population_ps"] >= min_population) & ~df["ref_area"].isin(excluded_areas)].copy()

    # The log is only defined for strictly positive values
    non_positive = (df[COLUMNS_WITH_LOG] <= 0).sum()
    if non_positive.any():
        raise ValueError(f"Non-positive values, log impossible: {non_positive[non_positive > 0].to_dict()}")

    for col in COLUMNS_WITH_LOG:
        df[f"log_{col}"] = np.log(df[col])
    return df


def standardize(df, features):
    """Z-score of each feature (mean 0, std 1), with country identifiers kept for reading."""
    z = (df[features] - df[features].mean()) / df[features].std()
    return df[["ref_area", "name"]].join(z)


def extreme_values(df_z, features, threshold=Z_THRESHOLD):
    """One row per (country, feature) with |z| > threshold, most extreme first."""
    long = df_z.melt(id_vars=["ref_area", "name"], value_vars=features, var_name="feature", value_name="z")
    return long[long["z"].abs() > threshold].sort_values("z", key=abs, ascending=False)


def farthest_countries(df_z, features, n=5):
    """Countries farthest from the center of the cloud: Euclidean norm of their z-score vector (what k-means sees)."""
    distance = np.sqrt((df_z[features] ** 2).sum(axis=1))
    return df_z[["ref_area", "name"]].assign(distance=distance).nlargest(n, "distance")

## Standardisation

In [ ]:
df_log = prepare_data()

ACTIVE_FEATURES = [
    "log_gdp_per_capita_ppp_current_ppp",
    "life_expectancy_yr",
    "log_co2_emissions_per_capita_t",
    "log_pm2_5_exposure_mcg_m3",
    "urban_population_percentage_pt",
]

df_standardized = standardize(df_log, ACTIVE_FEATURES)
df_standardized

In [5]:
df_standardized.describe().T

,count,mean,std,min,25%,50%,75%,max
log_gdp_per_capita_ppp_current_ppp,162.0,-4.276415e-16,1.0,-2.218231,-0.722306,0.065974,0.874760,1.858157
life_expectancy_yr,162.0,-5.811538e-16,1.0,-2.572115,-0.799540,0.114697,0.754166,1.508435
log_co2_emissions_per_capita_t,162.0,7.675616e-17,1.0,-2.633551,-0.677974,0.210873,0.734819,2.228335
log_pm2_5_exposure_mcg_m3,162.0,1.918904e-16,1.0,-2.275076,-0.591664,0.003754,0.732025,2.663157
urban_population_percentage_pt,162.0,-1.973730e-16,1.0,-2.127378,-0.833196,0.063297,0.817304,1.797974


In [ ]:
# Univariate check: countries extreme on one feature
extreme_values(df_standardized, ACTIVE_FEATURES)

In [ ]:
# Multivariate check: countries far from the center on all features combined
farthest_countries(df_standardized, ACTIVE_FEATURES)

### Extremes check (main version, standardized, 162 countries)

**Observation.** Only 3 countries exceed |z| > 2.5, each on a single feature: Nigeria (life expectancy, −2.57), Somalia (log CO₂, −2.63), Qatar (log PM2.5, +2.66). On all features combined, Qatar is the farthest from the center (distance 4.4, high on every feature), followed by Niger, Central African Republic, Burundi and Chad (≈ 4.0).

**Conclusion.** The log transform worked: no extreme value on several features at once, all countries kept. Qatar is the most isolated country of the cloud: watch whether it forms a cluster on its own.

## Variants (session 06)

| Variant | Change from the main version |
|---|---|
| A | + health % of GDP |
| B | Pollution only: log CO₂ + log PM2.5 |
| C | CO₂ and PM2.5 without log |
| D | 177 countries: micro-states put back, except Nauru and Tuvalu (zero CO₂, log impossible) |
| E1 / E2 | − CO₂ / − urbanization |

_Note: the z-score of a feature only depends on its own column and on the set of countries. Variants B, E1 and E2 therefore show the same univariate extremes as the main version; only the multivariate distances change._

In [ ]:
VARIANTS = {
    "A - with health %": ACTIVE_FEATURES + ["health_expenditure_percentage_of_gdp_pt_gdp"],
    "B - pollution only": ["log_co2_emissions_per_capita_t", "log_pm2_5_exposure_mcg_m3"],
    "C - raw CO2 and PM2.5": [
        "log_gdp_per_capita_ppp_current_ppp",
        "life_expectancy_yr",
        "co2_emissions_per_capita_t",
        "pm2_5_exposure_mcg_m3",
        "urban_population_percentage_pt",
    ],
    "E1 - without CO2": [f for f in ACTIVE_FEATURES if f != "log_co2_emissions_per_capita_t"],
    "E2 - without urbanization": [f for f in ACTIVE_FEATURES if f != "urban_population_percentage_pt"],
}

for variant, features in VARIANTS.items():
    df_z = standardize(df_log, features)
    print(f"=== Variant {variant} ({len(features)} features)")
    display(extreme_values(df_z, features))
    display(farthest_countries(df_z, features))

In [ ]:
# Variant D: micro-states put back, except Nauru and Tuvalu (zero CO₂ since 2000, log impossible)
ZERO_CO2_AREAS = ["NRU", "TUV"]
df_log_d = prepare_data(min_population=0, excluded_areas=ZERO_CO2_AREAS)

df_z_d = standardize(df_log_d, ACTIVE_FEATURES)
print(f"=== Variant D - micro-states back ({len(df_log_d)} countries)")
display(extreme_values(df_z_d, ACTIVE_FEATURES))
display(farthest_countries(df_z_d, ACTIVE_FEATURES))

### Extremes check (variants)

**Observation.** _To fill._

**Conclusion.** _To fill._